# COFOG-Panel: Harmonizing IMF GFS COFOG Data into Reproducible Panel Datasets

This notebook automates the setup, installation, and execution of the COFOG-Panel pipeline based on the project [README](README.md).

In [ ]:
# @title 1. Setup Environment and Install Dependencies
# @markdown This cell installs dependencies and sets up the package in editable mode.

# Install dependencies from requirements.txt
!pip install -r requirements.txt

# Install the package in editable mode to use the `cofog-panel` CLI command
!pip install -e .

## 2. Prepare Data

Ensure your input files are placed in the `./data/` directory:
- `gfs_raw_data.xlsx` (Main COFOG source)
- `country_codes.xlsx` (Country mapping file)

In [ ]:
import os

# Create necessary directories
os.makedirs('data', exist_ok=True)
os.makedirs('output', exist_ok=True)
os.makedirs('intermediate_splits', exist_ok=True)

print("Directories verified.")

## 3. Run Pipeline Stages

You can run the stages individually or use the orchestrator.

In [ ]:
# @title Orchestrator: Run Full Pipeline
# @markdown This command automates Stages 1 through 5 sequentially.

!cofog-panel run \
    --source-file ./data/gfs_raw_data.xlsx \
    --lookup-file ./data/country_codes.xlsx \
    --cofog "Defence" \
    --sector "General government" \
    --output-cols "DATA_DEFENCE"

### Individual Stages (Manual Execution)

In [ ]:
# Stage 1 & 2: Validation
!cofog-panel check-format --source-file ./data/gfs_raw_data.xlsx
!cofog-panel check-country-format --lookup-file ./data/country_codes.xlsx

# Stage 3: Seeding the Master Schema
!cofog-panel seed-master --lookup-file ./data/country_codes.xlsx --output-dir ./output

# Stage 4: ETL and Data Splitting
!cofog-panel split --source-file ./data/gfs_raw_data.xlsx --cofog "Defence" --filter-type "Percent of GDP"

# Stage 5: Harmonization and Aggregation
!cofog-panel aggregate --master-file ./output/COFOG_MASTER_SCHEMA.xlsx --folder-path ./intermediate_splits --data-col "DATA_DEFENCE" --sector "Local Government"